In [0]:
# for colab
%tensorflow_version 2.x

In [0]:
import os
import tensorflow as tf
import numpy as np

In [0]:
# for colab
from google.colab import drive
drive.mount('/content/drive')

In [0]:
os.chdir("drive/My Drive/Colab Notebooks")

In [0]:
# that's the file provided in the assignment
# I put it in my Colab Notebooks folder
!python3 prepare_data.py shakespeare_input.txt shake 

# you can try running the script with the -h flag to get help on the parameters

In [0]:
from prepare_data import parse_seq
import pickle

# this is just a datasets of "bytes" (not understandable)
data = tf.data.TFRecordDataset("shake.tfrecords")

# this maps a parser function that properly interprets the bytes over the dataset
data = data.map(lambda x: parse_seq(x, 200))

# a map from characters to indices
vocab = pickle.load(open("shake_vocab", mode="rb"))
vocab_size = len(vocab)
# inverse mapping: indices to characters
ind_to_ch = {ind: ch for (ch, ind) in vocab.items()}

print(vocab)
print(vocab_size)

In [0]:
# 1st, the way the data is stored: sequence of indices
# 2nd, each index converted to a one-hot vector (can't see this very well...)
# 3rd just for show, we can invert the one-hot conversion via argmax
# 4th, each index converted to character
# 5th, make it a bit prettier
for elem in data:
    print(elem, "\n")
    oh = tf.one_hot(elem, depth=vocab_size)
    print(oh, "\n")
    print(tf.argmax(oh, axis=-1), "\n")
    as_chars = [ind_to_ch[ind] for ind in elem.numpy()]
    print(as_chars, "\n")
    print("".join(as_chars), "\n")
    input()

In [0]:
# remember, for training we should batch data etc.
train_data = data.shuffle(30000).batch(128).repeat()

# each batch is shape batch_size x seq_len x vocab_size
for batch in train_data:
    oh_batch = tf.one_hot(batch, depth=vocab_size)
    print(oh_batch.shape)
    input()

In [0]:
# now, the task is:
# build an RNN that outputs p(x(t+1) | x1,x2,...,xt)
# in an RNN, this actually reduces to p(x(t+1) | ht) where ht is the current state

# we can interpret the task like this: we have an input sequence and an output
# sequence; at each time step, predict the next character in the output from
# the input so far.

# e.g. say x = "I have a dog."

# then input: "I have a dog"
# and target: " have a dog."

# this is a classification task!! each character represents a class, the RNN
# should produce a softmax output, and you can use cross-entropy loss.
# It is trained so that the RNN outputs the "correct" next character.
# it's basically just like MNIST :) except you have an additional axis (time).